In [ ]:
import numpy as np
import cv2
import glob

def calibrate_camera(image_dir, chessboard_size=(9, 6), square_size_mm=25.0):
    objpoints = []
    imgpoints = []

    objp = np.zeros((chessboard_size[0] * chessboard_size[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:chessboard_size[0], 0:chessboard_size[1]].T.reshape(-1, 2)
    objp *= square_size_mm

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

    # Advanced flags to improve detection on screens
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE + cv2.CALIB_CB_FAST_CHECK

    images = glob.glob(f"{image_dir}/*.jpg")

    if not images:
        print("No images found.")
        return None, None

    successes = 0
    gray = None

    for fname in images:
        img = cv2.imread(fname)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        ret, corners = cv2.findChessboardCorners(gray, chessboard_size, flags)

        if ret:
            objpoints.append(objp)
            corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
            imgpoints.append(corners2)
            successes += 1

    if successes == 0:
        print("Calibration failed: Chessboard not detected in any image.")
        return None, None

    ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

    print(f"Calibration successful ({successes}/{len(images)} images used).")
    print(f"Intrinsic Matrix (K):\n{np.round(K, 2)}")
    print(f"Distortion Coefficients:\n{np.round(dist.flatten(), 5)}")

    total_error = 0
    for i in range(len(objpoints)):
        projected_imgpoints, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, dist)
        error = cv2.norm(imgpoints[i], projected_imgpoints, cv2.NORM_L2) / len(projected_imgpoints)
        total_error += error

    print(f"Mean Reprojection Error: {total_error / len(objpoints):.4f} px")

    return K, dist

K_real, dist_real = calibrate_camera("calibracao", chessboard_size=(9, 6), square_size_mm=15.0)